# Model Results of Diabetes Classification Pipeline

Evaluation of two models trained and evaluated on GCP Dataproc: **Logistic Regression** and **Random Forest** .

| Section | Content |
|---|---|
| 1. Logistic Regression Probability Calibration | Effect of Platt scaling on probability estimates |
| 2. Model Performance | Confusion matrix, AUC comparison, sensitivity & specificity, risk bands |
| 3. Feature Importance | Random Forest Gini-based feature ranking |
| 4. Risk by Age | Predicted vs actual diabetic rate across age groups |
| 5. Risk by BMI | Predicted vs actual diabetic rate across BMI categories |
| 6. Risk by Binary Factors | Hypertension and heart disease risk lift |

## Imports & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlepad': 10})

EVAL_DIR   = os.path.join('..', 'evaluation')
AGG_DIR    = os.path.join('..', 'aggregations')
IMAGES_DIR = 'images'
os.makedirs(IMAGES_DIR, exist_ok=True)

lr_raw      = pd.read_csv(os.path.join(EVAL_DIR, 'evaluation_metrics_lr.csv')).iloc[0]
rf_raw      = pd.read_csv(os.path.join(EVAL_DIR, 'evaluation_metrics_rf.csv')).iloc[0]
fi          = pd.read_csv(os.path.join(EVAL_DIR, 'evaluation_feature_importance_rf.csv'))
agg         = pd.read_csv(os.path.join(AGG_DIR,  'evaluation_risk_aggregation_lr.csv'))
risk_summary = pd.read_csv(os.path.join(AGG_DIR, 'evaluation_risk_summary_lr.csv'))

age_df     = agg[agg['dimension'] == 'age'].copy()
bmi_df     = agg[agg['dimension'] == 'bmi'].copy()
htn_df     = agg[agg['dimension'] == 'hypertension'].copy()
hd_df      = agg[agg['dimension'] == 'heart_disease'].copy()
hba1c_df   = agg[agg['dimension'] == 'hbA1c_level'].copy()
glucose_df = agg[agg['dimension'] == 'blood_glucose_level'].copy()

# Risk stratification thresholds (cal set p80 / p90) — from 03_04_risk.py logs
RISK_THRESH_MEDIUM = 0.0456   # 80th percentile of calibration set prob_diabetic
RISK_THRESH_HIGH   = 0.3562   # 90th percentile of calibration set prob_diabetic

print('All data loaded.')

## 1. Calibration: Correcting Class-Weight Inflation

Becuase the data is highly imbalance at 8.5 % diabetes prevalence, we applied class weighting, which inflates raw LR predicted probabilities: the model outputs an average of ~21.5% when true prevalence is ~8.2%, a 2.6× overestimate. 

Platt scaling (a second logistic regression fitted on a held-out calibration set) corrects this, pulling the average predicted probability down to match true prevalence. After calibration, a score of 0.20 genuinely means ~20% estimated risk.

In [ ]:
# Calibration statistics logged by 03_02_lr_calibration.py and 03_03_lr_apply_calibration.py
PRE_CAL   = 0.215   # raw LR average predicted probability (inflated by class weighting)
POST_CAL  = 0.082   # after Platt scaling
TRUE_PREV = 0.082   # true diabetes prevalence in test set

fig, ax = plt.subplots(figsize=(7, 5))

labels = ['Pre-Calibration\n(Raw LR)', 'Post-Calibration\n(Platt Scaled)', 'True Prevalence\n(Ground Truth)']
vals   = [PRE_CAL, POST_CAL, TRUE_PREV]
colors = ['#e74c3c', '#27ae60', '#2c3e50']
bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor='white', width=0.45)

for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004,
            f'{v:.1%}', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.axhline(TRUE_PREV, color='#2c3e50', linestyle='--', linewidth=1.4, alpha=0.5)
ax.set_ylim(0, 0.28)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylabel('Average Predicted Probability')
ax.set_title('Effect of Calibration on Logistic Regression Prediction', fontsize=13, fontweight='bold')
ax.set_axisbelow(True)

mid_x = bars[0].get_x() + bars[0].get_width() / 2
ax.annotate('', xy=(mid_x, TRUE_PREV + 0.004), xytext=(mid_x, PRE_CAL - 0.004),
            arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=1.8))
ax.text(mid_x + 0.17, (PRE_CAL + TRUE_PREV) / 2,
        '2.6× over-\nestimate', fontsize=9, color="#25201f", fontweight='bold', va='center')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'calibration.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: calibration.png')


## 2. Model Performance

### 2a. Confusion Matrix - Logistic Regression

Confusion matrix on the 20% held-out test set (total n ≈ 20,000 records; percentages shown are shares of that total). The threshold (0.31) was tuned to prioritize recall as missing a diabetic case carries a higher cost than a false positive referral. 

In [ ]:
tn, fp = int(lr_raw['tn']), int(lr_raw['fp'])
fn, tp = int(lr_raw['fn']), int(lr_raw['tp'])
total  = tn + fp + fn + tp

cm = np.array([[tn, fp],
               [fn, tp]])

# Dark blue = correct, light blue-grey = wrong
cell_colors = np.array([['#2980b9', '#aed6f1'],
                         ['#aed6f1', '#2980b9']])
text_colors = np.array([['white',   '#2c3e50'],
                         ['#2c3e50', 'white']])

cell_labels = np.array([
    ['True Negative',  'False Positive'],
    ['False Negative', 'True Positive'],
])
cell_desc = np.array([
    ['Correctly told:\nno diabetes',        'Wrongly flagged:\nnot diabetic, but flagged'],
    ['Missed case:\ndiabetic, not caught',  'Correctly identified:\nhas diabetes'],
])

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_aspect('equal')
ax.axis('off')

for row in range(2):
    for col in range(2):
        x, y  = col, 1 - row
        count = cm[row, col]
        pct   = count / total * 100
        tc    = text_colors[row, col]

        rect = plt.Rectangle((x, y), 1, 1, color=cell_colors[row, col], alpha=0.9)
        ax.add_patch(rect)

        ax.text(x + 0.5, y + 0.72, cell_labels[row, col],
                ha='center', va='center', fontsize=11, fontweight='bold', color=tc)
        ax.text(x + 0.5, y + 0.50, f'{count:,}  ({pct:.1f}%)',
                ha='center', va='center', fontsize=14, fontweight='bold', color=tc)
        ax.text(x + 0.5, y + 0.22, cell_desc[row, col],
                ha='center', va='center', fontsize=8.5, color=tc, alpha=0.85,
                linespacing=1.4)

for col, label in enumerate(['Predicted:\nNo Diabetes', 'Predicted:\nHas Diabetes']):
    ax.text(col + 0.5, 2.08, label, ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='#2c3e50')

for row, label in enumerate(['Actual:\nNo Diabetes', 'Actual:\nHas Diabetes']):
    ax.text(-0.08, 1 - row + 0.5, label, ha='right', va='center',
            fontsize=11, fontweight='bold', color='#2c3e50')


plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'confusion_matrix_lr.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_lr.png')


### 2b. Sensitivity & Specificity — Logistic Regression

At the tuned threshold (0.31), the model correctly identifies 77.4% of diabetic records (sensitivity) and clears 95.2% of non-diabetic records (specificity). The trade-off is intentional: class weighting shifts the decision boundary to favor recall over precision, reducing missed diagnoses at the cost of more false positives.

In [ ]:
sensitivity = float(lr_raw['sensitivity'])
specificity = float(lr_raw['specificity'])

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (label, value, desc, color) in zip(axes, [
    ('Sensitivity', sensitivity,
     f'Of every 100 diabetic records,\nthe model correctly\nidentifies {sensitivity*100:.0f} of them.',
     '#2980b9'),
    ('Specificity', specificity,
     f'Of every 100 non-diabetic records,\nthe model correctly\nclears {specificity*100:.0f} of them.',
     '#2980b9'),
]):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Background card
    rect = plt.Rectangle((0.05, 0.05), 0.9, 0.9, color=color, alpha=0.1, transform=ax.transAxes)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=color, alpha=0.08))

    # Big number
    ax.text(0.5, 0.68, f'{value:.1%}', ha='center', va='center',
            fontsize=42, fontweight='bold', color=color, transform=ax.transAxes)

    # Label
    ax.text(0.5, 0.88, label, ha='center', va='center',
            fontsize=16, fontweight='bold', color='#2c3e50', transform=ax.transAxes)

    # Plain-English description
    ax.text(0.5, 0.35, desc, ha='center', va='center',
            fontsize=11, color='#555', transform=ax.transAxes, linespacing=1.6)

    # Thin bottom bar showing the value visually
    ax.add_patch(plt.Rectangle((0.1, 0.12), 0.8, 0.06, color='#dde', transform=ax.transAxes))
    ax.add_patch(plt.Rectangle((0.1, 0.12), 0.8 * value, 0.06, color=color, alpha=0.8, transform=ax.transAxes))

plt.suptitle('Logistic Regression — Sensitivity & Specificity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'sensitivity_specificity.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sensitivity_specificity.png')

### 2c. Risk Stratification: Low / Medium / High

The calibrated LR score is bucketed into three risk bands using percentile cutoffs computed on the calibration set:

| Band | Threshold | n (test) | Avg Predicted Prob | Actual Diabetic Rate |
|---|---|---|---|---|
| Low | < 0.046 | 16,058 | 0.5% | 1.1% |
| Medium | 0.046 – 0.356 | 2,056 | 15.2% | 11.2% |
| High | ≥ 0.356 | 1,976 | 62.8% | 62.0% |

Thresholds are the 80th percentile (0.0456) and 90th percentile (0.3562) of `prob_diabetic` on the calibration set. 80% of test records fall into Low risk; the High band captures 62% actual positive rate among flagged records.

In [ ]:
band_order  = ['Low', 'Medium', 'High']
band_colors = ['#3498db', '#e67e22', '#e74c3c']

rs = risk_summary.set_index('risk_band').reindex(band_order).reset_index()
total_n = rs['n'].sum()
rs['pct'] = rs['n'] / total_n * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Left: record count per band
bars = axes[0].bar(rs['risk_band'], rs['n'], color=band_colors, edgecolor='white', width=0.5, alpha=0.88)
for bar, row in zip(bars, rs.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{row.n:,}\n({row.pct:.0f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Records per Risk Band', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Record Count')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].set_ylim(0, rs['n'].max() * 1.22)
axes[0].set_axisbelow(True)

# Middle: avg predicted prob vs actual diabetic rate
x, w = np.arange(3), 0.35
b1 = axes[1].bar(x - w/2, rs['avg_prob'],     w, label='Avg Predicted Prob',  color='#3498db', alpha=0.85)
b2 = axes[1].bar(x + w/2, rs['diabetic_rate'], w, label='Actual Diabetic Rate', color='#2ecc71', alpha=0.85)
for bars_pair in [b1, b2]:
    for bar in bars_pair:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.008,
                     f'{h:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(band_order)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].set_title('Predicted vs Actual Rate per Band', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Rate')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, rs['diabetic_rate'].max() * 1.3)
axes[1].set_axisbelow(True)

# Right: threshold scale diagram
ax3 = axes[2]
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')
ax3.set_title('Risk Band Thresholds', fontsize=12, fontweight='bold')

bar_y, bar_h = 0.38, 0.14
ax3.add_patch(plt.Rectangle((0.05, bar_y), RISK_THRESH_MEDIUM, bar_h,
                              color='#3498db', alpha=0.82))
ax3.add_patch(plt.Rectangle((0.05 + RISK_THRESH_MEDIUM, bar_y),
                              RISK_THRESH_HIGH - RISK_THRESH_MEDIUM, bar_h,
                              color='#e67e22', alpha=0.82))
ax3.add_patch(plt.Rectangle((0.05 + RISK_THRESH_HIGH, bar_y),
                              0.9 - RISK_THRESH_HIGH, bar_h,
                              color='#e74c3c', alpha=0.82))

for x_pos, label in [(0.05 + RISK_THRESH_MEDIUM / 2, 'Low'),
                      (0.05 + (RISK_THRESH_MEDIUM + RISK_THRESH_HIGH) / 2, 'Medium'),
                      (0.05 + (RISK_THRESH_HIGH + 0.9) / 2, 'High')]:
    ax3.text(x_pos, bar_y + bar_h / 2, label, ha='center', va='center',
             fontsize=11, fontweight='bold', color='white')

for x_pos, label in [(0.05 + RISK_THRESH_MEDIUM, f'{RISK_THRESH_MEDIUM}'),
                      (0.05 + RISK_THRESH_HIGH,   f'{RISK_THRESH_HIGH}')]:
    ax3.axvline(x=x_pos, ymin=bar_y - 0.08, ymax=bar_y + bar_h + 0.08,
                color='#2c3e50', linewidth=0, alpha=0)
    ax3.annotate('', xy=(x_pos, bar_y - 0.05), xytext=(x_pos, bar_y),
                 arrowprops=dict(arrowstyle='-', color='#2c3e50', lw=1.5))
    ax3.text(x_pos, bar_y - 0.12, label, ha='center', va='top',
             fontsize=9, color='#2c3e50', fontweight='bold')

ax3.text(0.5, 0.78, 'prob_diabetic', ha='center', fontsize=10, color='#555')
ax3.text(0.05, 0.25, '0.0', ha='center', fontsize=9, color='#888')
ax3.text(0.95, 0.25, '1.0', ha='center', fontsize=9, color='#888')
ax3.text(0.5, 0.12, 'Thresholds: p80 = 0.0456  |  p90 = 0.3562',
         ha='center', fontsize=9, color='#555', style='italic')

plt.suptitle('LR Risk Stratification — Test Set (n ≈ 20,000)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'risk_stratification.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: risk_stratification.png')

## 3. Feature Importance: Random Forest

Gini-based feature importance extracted directly from the trained model. Each score reflects the proportion of total impurity reduction attributed to that feature across all decision trees.

Clinical lab values dominate: blood glucose level (41.8%) and HbA1c (41.6%) together account for ~83% of total importance. Age is the next strongest signal (13.3%), followed by BMI (2.7%). Comorbidity flags (hypertension, heart disease) contribute under 1% each, their univariate association with diabetes largely disappears once lab values are present in the model.

In [ ]:
from matplotlib.patches import Patch

NAME_MAP = {
    'blood_glucose_level': 'Blood Glucose Level',
    'hbA1c_level':         'HbA1c Level',
    'age':                 'Age',
    'bmi':                 'BMI',
    'hypertension':        'Hypertension',
    'heart_disease':       'Heart Disease',
}

def feature_color(name):
    if name in ('Blood Glucose Level', 'HbA1c Level'): return '#9b59b6'
    if name == 'Age':  return '#3498db'
    if name == 'BMI':  return '#27ae60'
    return '#e74c3c'

fi_plot = (
    fi[fi['feature'].isin(NAME_MAP)]
      .assign(label=lambda d: d['feature'].map(NAME_MAP))
      .query('importance > 0')
      .sort_values('importance')
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [feature_color(lbl) for lbl in fi_plot['label']]
bars = ax.barh(fi_plot['label'], fi_plot['importance'], color=colors, alpha=0.85, edgecolor='white')

for bar in bars:
    w = bar.get_width()
    if w > 0.001:
        ax.text(w + 0.005, bar.get_y() + bar.get_height()/2,
                f'{w:.4f}', va='center', fontsize=10)

ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Random Forest — Feature Importance Ranking', fontsize=13, fontweight='bold')
ax.set_xlim(0, fi_plot['importance'].max() * 1.18)
ax.set_axisbelow(True)

legend_elements = [
    Patch(facecolor='#9b59b6', alpha=0.85, label='Clinical lab values'),
    Patch(facecolor='#3498db', alpha=0.85, label='Age'),
    Patch(facecolor='#27ae60', alpha=0.85, label='BMI'),
    Patch(facecolor='#e74c3c', alpha=0.85, label='Comorbidity flags'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')


##  4. AUC Metrics: LR vs Random Forest

AUC-ROC and AUC-PR compared side-by-side. RF achieves slightly higher discrimination (AUC-ROC 0.967 vs 0.959), likely capturing non-linear interactions between lab values and age that LR cannot model.

AUC-PR is the more informative metric under class imbalance (8.5% positive class), both models hold above 0.80.

In [ ]:
lr_m = {
    'AUC-ROC': round(float(lr_raw['auc_roc']), 4),
    'AUC-PR':  round(float(lr_raw['auc_pr']),  4),
}
rf_m = {
    'AUC-ROC': round(float(rf_raw['auc_roc']), 4),
    'AUC-PR':  round(float(rf_raw['auc_pr']),  4),
}

fig, ax = plt.subplots(figsize=(7, 5))
x, w = np.arange(2), 0.35
bars_lr = ax.bar(x - w/2, list(lr_m.values()), w, label='Logistic Regression', color='#3498db', alpha=0.85)
bars_rf = ax.bar(x + w/2, list(rf_m.values()), w, label='Random Forest',       color='#e67e22', alpha=0.85)

for bars in [bars_lr, bars_rf]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(list(lr_m.keys()), fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score')
ax.set_title('AUC Metrics — Logistic Regression vs Random Forest', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'metrics_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: metrics_comparison.png')


## 5. Risk Distribution by Bucket Groups

Average LR calibrated probability vs actual diabetic rate per bucket group, alongside record counts. Predicted probabilities track actual rates closely, a sign that Platt scaling is working. 


In [ ]:
AGE_ORDER = ['Children (<18)', 'Young (18-34)', 'Middle (35-54)', 'Senior (>=55)']
GLUCOSE_ORDER = ['Normal (<100)', 'Prediabetes (100-125)', 'High (>=126)']
HBA1C_ORDER = ['Normal (<5.7)', 'Prediabetes (5.7-6.4)', 'Diabetic (>=6.5)']
BMI_ORDER = ['Underweight (<18.5)', 'Normal (18.5-24.9)', 'Overweight (25-29.9)', 'Obese (>=30)']


def plot_dimension(df, bucket_col, order, title, save_name):
    df_s = df.set_index(bucket_col).reindex(order).reset_index()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: predicted prob vs actual diabetic rate
    x, w = np.arange(len(order)), 0.35
    b1 = ax1.bar(x - w/2, df_s['avg_prob'],     w, label='Avg Predicted Prob', color='#3498db', alpha=0.85)
    b2 = ax1.bar(x + w/2, df_s['diabetic_rate'], w, label='Actual Diabetic Rate', color='#e74c3c', alpha=0.85)

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            if h > 0.003:
                ax1.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                         f'{h:.1%}', ha='center', va='bottom', fontsize=8)

    ax1.set_xticks(x)
    ax1.set_xticklabels(order, rotation=15, ha='right', fontsize=9)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax1.set_ylabel('Rate')
    ax1.set_title(f'Predicted vs Actual Diabetic Rate: {title}', fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.set_axisbelow(True)

    # Right: record counts
    b3 = ax2.bar(order, df_s['n'], color='#9b59b6', alpha=0.85, edgecolor='white')
    for bar in b3:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 30,
                 f'{int(h):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.set_ylabel('Record Count')
    ax2.set_title(f'Record Count: {title}', fontweight='bold')
    ax2.set_xticks(range(len(order)))
    ax2.set_xticklabels(order, rotation=15, ha='right', fontsize=9)
    ax2.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, save_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')




### 5a. Risk Distribution by Age Group

Average LR calibrated probability vs actual diabetic rate across age group. Risk scales monotonically with age: seniors (≥55) carry over 10× the predicted risk of those under 18.

In [ ]:
plot_dimension(age_df, 'bucket', AGE_ORDER, 'Age Group', 'risk_by_age.png')

### 5b. Risk Distribution by Blood Glucose Level

Average LR calibrated probability vs actual diabetic rate across blood glucose bands. Normal glucose stays low, while prediabetes and high glucose rise sharply, matching the expected metabolic gradient.


In [ ]:
plot_dimension(glucose_df, 'bucket', GLUCOSE_ORDER, 'Blood Glucose Level', 'risk_by_blood_glucose.png')


### 5c. Risk Distribution by HbA1c Level

Average LR calibrated probability vs actual diabetic rate across HbA1c bands. The model should step up clearly from normal to prediabetes and diabetic ranges, reflecting the stronger lab-value signal.

In [ ]:
plot_dimension(hba1c_df, 'bucket', HBA1C_ORDER, 'HbA1c Level', 'risk_by_hba1c.png')

### 5d. Risk Distribution by BMI Category

Predicted vs actual diabetic rate across four BMI bands. Records in the obese category average ~16.7% predicted probability versus ~0.6% for underweight — capturing the expected metabolic gradient. As with age, calibrated predictions closely match observed rates.

In [ ]:
plot_dimension(bmi_df, 'bucket', BMI_ORDER, 'BMI Category', 'risk_by_bmi.png')


### 5e. Risk by Binary Factors — Hypertension & Heart Disease

Predicted and actual diabetic rates split by hypertension and heart disease status. Both comorbidities roughly quadruple predicted risk: hypertension raises it from ~6.6% to ~27.2%; heart disease raises it from ~7.3% to ~30.5%. Despite this strong univariate signal, both features contribute under 1% to Random Forest importance — their effect is largely mediated by correlated lab values once the full feature set is used.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

datasets = [
    (htn_df, 'Hypertension',  {0: 'No HTN',           1: 'With HTN'}),
    (hd_df,  'Heart Disease',  {0: 'No Heart Disease', 1: 'Heart Disease'}),
]

for ax, (df, title, labels) in zip(axes, datasets):
    df2 = df.copy()
    df2['bucket'] = df2['bucket'].astype(int)
    df2['label']  = df2['bucket'].map(labels)
    df2 = df2.sort_values('bucket')

    x, w = np.arange(len(df2)), 0.35
    b1 = ax.bar(x - w/2, df2['avg_prob'],     w, label='Avg Predicted Prob',  color='#3498db', alpha=0.85, edgecolor='white')
    b2 = ax.bar(x + w/2, df2['diabetic_rate'], w, label='Actual Diabetic Rate', color='#e74c3c', alpha=0.85, edgecolor='white')

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                    f'{h:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(df2['label'], fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.set_ylim(0, max(df2['diabetic_rate'].max(), df2['avg_prob'].max()) * 1.35)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Rate')
    ax.legend(fontsize=8)
    ax.set_axisbelow(True)

plt.suptitle('Risk by Binary Factors: Hypertension & Heart Disease',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'risk_by_binary_factors.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: risk_by_binary_factors.png')


## 6. Conclusion

**Model Performance.**
Both models achieve strong AUC-ROC (~0.96) and AUC-PR (~0.81–0.83) on the 20% held-out test set (~20,000 records). For Logistic Regression, threshold-based metrics at the tuned threshold (0.31) show sensitivity of 77.35%, specificity of 95.21%, accuracy of 93.76%, and F1 of 0.67. Precision is modest (~59%) because class weighting was applied to prioritize recall — missing a diabetic case is more costly than a false positive referral on a dataset with 8.5% prevalence. The Random Forest achieves a slightly higher AUC-ROC (0.967 vs 0.959), but threshold-based metrics are not stored in the current pipeline output.

**Feature Importance.**
Clinical lab values dominate: blood glucose level (41.8%) and HbA1c (41.6%) together account for ~83% of total importance in the Random Forest. Age is the next strongest signal (13.3%), followed by BMI (2.7%). Comorbidity flags (hypertension, heart disease) each contribute less than 1% individually — though the aggregation results show they still shift risk considerably in practice.

**Risk Distribution.**
The aggregation results are consistent and calibrated (predicted probabilities closely track actual diabetic rates):

- **Age:** Seniors (≥55) have 18.1% avg predicted probability vs 0.5% for those under 18 — a ~40× difference. Risk scales monotonically with age.
- **BMI:** Records in the obese category average 16.7% predicted probability vs 0.6% for underweight — the model captures the expected metabolic gradient.
- **Binary factors:** Hypertension raises predicted risk from 6.6% to 27.2%; heart disease raises it from 7.3% to 30.5%. Both comorbidities roughly quadruple predicted risk.

**Key Takeaway.**
Lab values (blood glucose, HbA1c) are the primary model signal, reflecting clinical reality. Demographic and lifestyle factors (age, BMI, comorbidities) are secondary but actionable — they identify high-risk profiles that can guide preventive outreach before lab results are available.